# Внимание и маски

В лекции мы разобрали, как запрос сравнивается с ключами и превращает их значения в новый вектор. Теперь соберём это вычисление на NumPy. Начнём с маленьких матриц, а затем изменим доступные связи и посмотрим на результат.

Главный вопрос работы: если позиция не видит будущий токен, может ли изменение этого токена повлиять на её выход?

In [ ]:
from pathlib import Path
import os
import json
import re
import unicodedata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Находим корень раздачи независимо от места запуска Jupyter.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "data" / "requests.csv").exists())
DATA = ROOT / "data"
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
np.random.seed(42)
plt.rcParams.update({"font.family": "DejaVu Sans", "figure.figsize": (8, 4)})


## От оценок к весам

Для каждого запроса attention получает строку оценок. После softmax она превращается в веса, сумма которых равна единице. Разберём одну короткую строку, прежде чем обрабатывать целую матрицу.

In [ ]:
demo_scores = np.array([0., np.log(2.)])
demo_exp = np.exp(demo_scores)
print("После экспоненты:", demo_exp)
print("После нормировки:", demo_exp / demo_exp.sum())

Получились веса примерно 1/3 и 2/3. Каждую экспоненту разделили на сумму экспонент этой строки. Если запросов несколько, у каждого должен быть собственный знаменатель.

## 1. Softmax для нескольких запросов

Напишите `softmax_rows(scores)` для двумерной матрицы. Каждая строка относится к отдельному запросу; нормировать нужно вдоль ключей.

Чтобы большие оценки не приводили к переполнению экспоненты, сначала вычтите максимум каждой строки. Сравните результат для исходной матрицы и для той же матрицы, к которой прибавили 10000. Что произошло с весами?

Позже закрытые ключи будут обозначаться `-np.inf`; их можно принимать. Для `NaN`, `+inf` и строки из одних `-inf` вызывайте `ValueError`: в этих случаях нужного распределения не получится.

In [ ]:
def softmax_rows(scores):
    # Ваш код
    raise NotImplementedError


In [ ]:
scores = np.array([[1000., 1001., 1002.], [-2., 0., 1.]])
p = softmax_rows(scores)
assert np.isfinite(p).all() and np.allclose(p.sum(axis=1), 1)
assert np.allclose(p, softmax_rows(scores + 10000))
assert not np.isclose(p.sum(), 1)  # здесь две отдельные строки-распределения
try:
    softmax_rows([[-np.inf, -np.inf]])
except ValueError:
    pass
else:
    raise AssertionError("Полностью закрытая строка должна быть отклонена")

## 2. Собираем attention

Используйте написанный softmax, чтобы закончить `attention(Q, K, V, allow=None)`. В заготовке уже есть проверки форм и значений. Допишите вычисление оценок `Q @ K.T / sqrt(d_k)`, применение маски, весов и выхода.

В `allow` True означает разрешённый ключ. Закрытые оценки заменяются на `-np.inf` до softmax. Если маска не передана, доступны все ключи. Функция возвращает пару `(output, weights)`.

Проверка воспроизведёт пример из лекции: Q=K=I₂, V₁=(10;0), V₂=(0;20). Сопоставьте первый выход с ручным расчётом. Какая величина задаёт знаменатель √dₖ — число позиций или число координат ключа?

In [ ]:
def attention(Q, K, V, allow=None):
    Q, K, V = [np.asarray(x, dtype=float) for x in [Q, K, V]]
    if any(x.ndim != 2 or min(x.shape) == 0 for x in [Q, K, V]):
        raise ValueError("Нужны непустые матрицы")
    if Q.shape[1] != K.shape[1] or K.shape[0] != V.shape[0]:
        raise ValueError("Несогласованные размерности")
    if not all(np.isfinite(x).all() for x in [Q, K, V]):
        raise ValueError("Входы должны быть конечными")
    if allow is None:
        allow = np.ones((len(Q), len(K)), dtype=bool)
    allow = np.asarray(allow)
    if allow.dtype != bool or allow.shape != (len(Q), len(K)) or not allow.any(axis=1).all():
        raise ValueError("Некорректная маска")
    # Ваш код
    raise NotImplementedError


In [ ]:
Q = K = np.eye(2)
V = np.array([[10., 0.], [0., 20.]])
output, weights = attention(Q, K, V)
assert np.allclose(weights[0], [0.66976155, 0.33023845])
assert np.allclose(output[0], [6.69761549, 6.60476901])
# Несимметричный cross-пример: два запроса, три ключа, два признака значений.
o, a = attention([[1, 0], [1, 1]], [[1, 0], [0, 1], [2, -1]], [[1, 2], [3, 4], [5, 6]])
assert o.shape == (2, 2) and a.shape == (2, 3)
assert np.allclose(a.sum(axis=1), 1)

## 3. Закрываем будущее

В заготовке даны причинные маски для двух и трёх позиций. Диагональ разрешена. Примените двухпозиционную маску к Q, K и V, сохранив `masked_output` и `masked_weights`. Сравните первую и вторую строки с предыдущим результатом.

Затем проведите опыт с трёхпозиционной матрицей X. Создайте `V_changed = X.copy()` и прибавьте (100;−50) к строке с индексом 2. В этом опыте Q=K=X; сначала V=X, затем V=V_changed. Меняется только V. Сравните первые два выхода до и после изменения, сначала с `mask3`, затем без маски.

Прежде чем выполнять сравнение, предскажите, в каком из двух вариантов появится разница. Найдите на карте весов связь, которая объясняет результат.

In [ ]:
causal = np.tril(np.ones((2, 2), dtype=bool))
X = np.array([[1., 0.], [0., 1.], [1., 1.]])
mask3 = np.tril(np.ones((3, 3), dtype=bool))
# Ваш код
raise NotImplementedError

In [ ]:
assert np.allclose(masked_weights[0], [1, 0])
assert np.allclose(masked_output[0], [10, 0])
assert np.allclose(masked_output[1], output[1])
assert np.all(masked_weights[~causal] == 0)

## 4. Не все позиции содержат текст

Считайте последний ключ матрицы X заполнителем padding. Постройте `padding_allow`, закрыв этот ключ для всех запросов, и объедините её с `mask3` через `&`. Сохраните объединённую маску в `combined`, а полученные веса — в `padding_weights`.

Посмотрите на последний столбец. Что с ним произошло? Почему при этом строки весов по-прежнему складываются в единицу?

Проверьте ещё два случая: строку, где закрыты все ключи, и попытку заменить запрещённую оценку нулём. В первом случае нужен `ValueError`. Во втором обратите внимание на вес ключа, который вы собирались исключить.

In [ ]:
# Ваш код
raise NotImplementedError

In [ ]:
assert padding_weights.shape == (3, 3)
assert np.allclose(padding_weights.sum(axis=1), 1)
assert softmax_rows([[1, 0]])[0, 1] > 0

## 5. Меняем порядок строк

Для этого опыта уберите все маски. Сравните `attention(X, X, X)` с результатом для переставленных строк `X[permutation]`. Верните выходы к одному порядку и проверьте совпадение.

Теперь прибавьте матрицу Pos. При перестановке содержимого позиционные значения должны остаться на своих местах: сравниваются `X + Pos` и `X[permutation] + Pos`. Сохраните выходы как `with_position` и `permuted_with_position`.

Запишите максимальную разницу `np.max(np.abs(permuted_with_position - with_position[permutation]))` в `position_difference`. Почему результат теперь отличается? Что изменилось бы, если переставить целиком `X + Pos`?

In [ ]:
permutation = np.array([2, 0, 1])
Pos = np.array([[0., 0.], [0.5, 0.], [0., 0.5]])
# Ваш код
raise NotImplementedError

In [ ]:
assert position_difference > 1e-6

## Разные веса, одинаковые значения

Пусть обе строки V равны (1;2). Предскажите выход для весов (0,9;0,1) и (0,2;0,8), затем проверьте себя. Что можно узнать по карте весов, если не смотреть на сами значения?

In [ ]:
same_values = np.array([[1., 2.], [1., 2.]])
first_weights = np.array([0.9, 0.1])
second_weights = np.array([0.2, 0.8])
print(first_weights @ same_values)
print(second_weights @ same_values)

## Обсуждение результатов

Выберите два опыта и объясните их по шагам, с опорой на полученные числа. Можно взять изменение будущего значения и перестановку строк. Укажите, что именно вы изменили, что оставили прежним и почему получили такой выход.

Ответьте ещё на два вопроса: почему маскирование нулём не закрывает ключ и почему разные карты весов могут дать одинаковый результат?

Сдайте выполненный ноутбук с этими пояснениями. Перед сдачей перезапустите ядро и выполните все ячейки по порядку.